# 01 · Veri Temizliği

Bu notebook, `src/collect.py` ile ESPN'den toplanan ham yanıtları (`data/raw/espn/`) okur, rehberin 5. adımındaki kontrolleri sırayla uygular ve temizlenmiş tabloları `data/processed/` altına CSV olarak kaydeder.

| Çıktı | Bir satır | Kullanıldığı soru |
|---|---|---|
| `matches.csv` | normal sezonda oynanmış bir maç | 1 (ev sahibi avantajı) |
| `standings.csv` | bir takımın sezon sonu puan tablosu satırı | 2, 3 |
| `top_scorers.csv` | bir lig-sezonun gol kralı (eşitlikte birden çok) | 3 |
| `players.csv` | bir takımın sezon kadro listesindeki bir oyuncu | 2 |

Kapsam 9 lig ve 2015/16–2025/26 sezonları. Sezon numarası başlangıç yılıdır (2019 = 2019/20). Her adımda yapılan gözlem ve alınan karar hücrelerin altında not edilmiştir.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
import parse  # ham JSON -> düz tablo
from collect import LEAGUES

PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
SEASONS = range(2015, 2026)


def keys(df, team_col="team_id"):
    # (lig, sezon, takım) üçlüleri; tablolar arası tutarlılık kontrolü için
    return set(zip(df["league"], df["season"], df[team_col]))

## 1. Maçlar

In [2]:
matches_raw = parse.matches()
print(matches_raw.shape)
matches_raw.head()

(39892, 16)


,league,season,season_type,season_slug,match_id,date,status,completed,home_team_id,home_team,away_team_id,away_team,home_goals,away_goals,attendance,has_details
0,eng.1,2014,5814,2014-2015-barclays-premier-league,395585,2015-01-01T12:45Z,STATUS_FULL_TIME,True,336,Stoke City,360,Manchester United,1,1,27203,True
1,eng.1,2014,5814,2014-2015-barclays-premier-league,395583,2015-01-01T15:00Z,STATUS_FULL_TIME,True,362,Aston Villa,384,Crystal Palace,0,0,29047,True
2,eng.1,2014,5814,2014-2015-barclays-premier-league,395584,2015-01-01T15:00Z,STATUS_FULL_TIME,True,306,Hull City,368,Everton,2,0,23865,True
3,eng.1,2014,5814,2014-2015-barclays-premier-league,395778,2015-01-01T15:00Z,STATUS_FULL_TIME,True,364,Liverpool,375,Leicester City,2,2,44720,True
4,eng.1,2014,5814,2014-2015-barclays-premier-league,395586,2015-01-01T15:00Z,STATUS_FULL_TIME,True,382,Manchester City,366,Sunderland,3,2,45367,True


### 1.1 Tekrar eden kayıtlar

In [3]:
matches_raw.duplicated("match_id").sum()

np.int64(0)

Tekrar eden maç yok. Her maç yalnızca bir takvim yılı dosyasında bulunuyor.

### 1.2 Veri tipleri

In [4]:
matches_raw.dtypes

league          object
season           int64
season_type      int64
season_slug     object
match_id        object
date            object
status          object
completed         bool
home_team_id    object
home_team       object
away_team_id    object
away_team       object
home_goals      object
away_goals      object
attendance       int64
has_details       bool
dtype: object

Skorlar, id'ler ve tarih metin olarak geliyor. Sayıya ve tarihe dönüştürülüyor.

In [5]:
matches = matches_raw.copy()
matches["date"] = pd.to_datetime(matches["date"], utc=True)
for col in ["match_id", "home_team_id", "away_team_id", "home_goals", "away_goals"]:
    matches[col] = pd.to_numeric(matches[col])
matches.dtypes

league                       object
season                        int64
season_type                   int64
season_slug                  object
match_id                      int64
date            datetime64[ns, UTC]
status                       object
completed                      bool
home_team_id                  int64
home_team                    object
away_team_id                  int64
away_team                    object
home_goals                    int64
away_goals                    int64
attendance                    int64
has_details                    bool
dtype: object

### 1.3 Sezon aralığı ve normal sezon

Maçlar takvim yılına göre çekildiği için tablo, kapsam dışındaki 2014/15 ve 2026/27 sezonlarının maçlarını da içeriyor. Ayrıca play-off ve baraj maçları normal sezona karışmış. ESPN bunları ayrı bir sezon tipiyle (`season_type`) işaretliyor. Her lig-sezonda en çok maçı olan tip normal sezon kabul ediliyor.

In [6]:
matches = matches[matches["season"].isin(SEASONS)]
main_type = matches.groupby(["league", "season"])["season_type"].agg(lambda s: s.value_counts().idxmax())
is_regular = matches["season_type"] == matches.set_index(["league", "season"]).index.map(main_type)
matches.loc[~is_regular, "season_slug"].value_counts()

season_slug
promotion-semifinals                         28
europa-conference-playoffs---semifinals      14
europa-league-playoffs---semi-finals         12
semifinals                                   12
europa-league-playoffs---final                8
promotion-final                               7
europa-conference-playoffs---final            7
final                                         4
semi-finals                                   4
europa-league-playoffs---semifinals           4
promotionrelegation-playoff-final             2
promotionrelegation-playoffs                  2
conference-league-playoffs---semifinals       2
promotion-playoff-finals                      2
promotion-playoff-semifinals                  1
promotionrelegation-playoff-quarterfinals     1
relegation-playoff                            1
promotion-playoff-quarterfinals               1
promotionrelegation-playoff-semifinals        1
conference-league-playoffs---final            1
Name: count, dtype: int64

Ayrılan maçlar Championship terfi play-off'ları, Eredivisie Avrupa play-off'ları ve Serie A ile Ligue 1 baraj maçları. Bunlar çıkarılıyor.

Kontrol: Çift devreli bir ligde normal sezonun maç sayısı takım × (takım − 1) olmalı. Bu sayıma henüz oynanmamış maçlar da dahil; onlar bir sonraki adımda ele alınıyor.

In [7]:
matches = matches[is_regular]
per_season = matches.groupby(["league", "season"]).agg(maç=("match_id", "size"), takım=("home_team_id", "nunique"))
per_season["beklenen"] = per_season["takım"] * (per_season["takım"] - 1)
per_season[per_season["maç"] != per_season["beklenen"]]

,,maç,takım,beklenen
league,season,,,


Farkı olan lig-sezon yok. Play-off'lar doğru ayrılmış.

### 1.4 Oynanmamış ve bitmemiş maçlar

In [8]:
matches.loc[~matches["completed"]].groupby(["league", "season", "status"]).size()

league  season  status            
bra.1   2016    STATUS_CANCELED         1
        2021    STATUS_SECOND_HALF      1
eng.2   2015    STATUS_SCHEDULED        2
        2022    STATUS_SCHEDULED       12
esp.1   2022    STATUS_SCHEDULED        5
fra.1   2016    STATUS_ABANDONED        1
        2019    STATUS_CANCELED       101
        2025    STATUS_ABANDONED        1
ned.1   2015    STATUS_SCHEDULED        2
        2019    STATUS_CANCELED        74
por.1   2021    STATUS_ABANDONED        1
        2022    STATUS_SCHEDULED        5
dtype: int64

Tamamlanmamış görünen 206 maç üç gruba ayrılıyor:

- **Hiç oynanmamış maçlar (`STATUS_CANCELED`, 176 maç):** Ligue 1 2019/20'de 101, Eredivisie 2019/20'de 74 maç pandemi nedeniyle iptal edildi. Brasileirão 2016'daki 1 maç, Chapecoense'nin uçak kazası nedeniyle oynanmadı.
- **Yarıda kalan maçlar (`STATUS_ABANDONED`, 3 maç):** Örneğin Primeira Liga 2021/22'deki Belenenses–Benfica maçı, Belenenses'te sahada 7'den az oyuncu kalınca bırakıldı.
- **ESPN'de durumu güncellenmemiş maçlar (`STATUS_SCHEDULED` 26, `STATUS_SECOND_HALF` 1):** Bu maçlar gerçekte oynandı ama ESPN'de kesin skoru yok.

**Karar:** Tamamlanmamış maçların hepsi çıkarılıyor. Durumu güncellenmemiş 27 maç toplamın binde birinden az.

In [9]:
matches = matches[matches["completed"]].copy()
len(matches)

36642

### 1.5 Eksik değerler

In [10]:
matches.isna().sum()

league          0
season          0
season_type     0
season_slug     0
match_id        0
date            0
status          0
completed       0
home_team_id    0
home_team       0
away_team_id    0
away_team       0
home_goals      0
away_goals      0
attendance      0
has_details     0
dtype: int64

Hiçbir sütunda eksik değer yok. Ancak seyirci sütununda eksik veri 0 olarak kodlanmış; bu bir sonraki adımda ele alınıyor.

### 1.6 Seyirci sayısındaki 0 değerleri

ESPN'de seyirci sayısının 0 olması iki anlama gelebiliyor: maç gerçekten seyircisiz oynandı ya da veri girilmedi. Pandemi dönemi dışında 0 görünen maçlar ikinci gruba giriyor.

**Karar:** Mart 2020 – Mayıs 2022 arasında seyirci sayısı 0 olan maçlar seyircisiz kabul ediliyor. Bu dönem dışındaki 0'lar eksik veri sayılıp NaN yapılıyor.

In [11]:
PANDEMIC_START = pd.Timestamp("2020-03-01", tz="UTC")
PANDEMIC_END = pd.Timestamp("2022-06-01", tz="UTC")  # Mayıs 2022 dahil
in_pandemic = (matches["date"] >= PANDEMIC_START) & (matches["date"] < PANDEMIC_END)
zero = matches["attendance"] == 0
pd.DataFrame({
    "pandemi döneminde 0": (zero & in_pandemic).groupby(matches["league"]).sum(),
    "pandemi dışında 0": (zero & ~in_pandemic).groupby(matches["league"]).sum(),
})

,pandemi döneminde 0,pandemi dışında 0
league,,
bra.1,597,261
eng.1,440,19
eng.2,638,4
esp.1,485,2
fra.1,322,9
ger.1,369,1
ita.1,510,12
ned.1,348,6
por.1,391,4


Pandemi dışındaki 0'ların büyük kısmı (261 maç) Brasileirão'da, özellikle 2016 ve 2018 sezonlarında. Diğer liglerde 11 sezon boyunca 1 ile 19 arasında maç var.

In [12]:
matches["attendance"] = matches["attendance"].astype("Int64")
matches.loc[zero & ~in_pandemic, "attendance"] = pd.NA
matches["attendance"].isna().sum()

np.int64(318)

### 1.7 Aykırı değerler

In [13]:
matches[["home_goals", "away_goals", "attendance"]].describe()

,home_goals,away_goals,attendance
count,36642.000000,36642.000000,36324.0
mean,1.513891,1.192020,22308.503359
std,1.282735,1.142992,18170.885787
min,0.000000,0.000000,0.0
25%,1.000000,0.000000,9758.5
50%,1.000000,1.000000,18615.0
75%,2.000000,2.000000,30952.25
max,10.000000,13.000000,146013.0


In [14]:
matches.nlargest(3, "attendance")[["league", "season", "home_team", "away_team", "attendance"]]

,league,season,home_team,away_team,attendance
8214,eng.2,2021,Huddersfield Town,Reading,146013
15404,esp.1,2015,Barcelona,Real Madrid,98902
15612,esp.1,2016,Barcelona,Real Madrid,98485


Huddersfield–Reading maçında görünen 146.013 seyirci gerçek olamaz, çünkü Huddersfield'ın stadyumu yaklaşık 24 bin kişilik. Bu liglerdeki en büyük stadyum yaklaşık 99 bin kişilik Camp Nou. İkinci ve üçüncü sıradaki Barcelona–Real Madrid maçları gerçekçi.

Takımın kendi sezon ortalamasının çok üstündeki değerlere de bakılıyor:

In [15]:
home_median = (matches[matches["attendance"] > 0]
               .groupby(["league", "season", "home_team_id"])["attendance"].transform("median"))
ratio = matches["attendance"] / home_median
print((ratio > 2).sum(), "maçta seyirci, ev sahibinin sezon medyanının 2 katından fazla")
matches.assign(oran=ratio.round(1)).nlargest(6, "oran")[["league", "season", "home_team", "away_team", "attendance", "oran"]]

597 maçta seyirci, ev sahibinin sezon medyanının 2 katından fazla


,league,season,home_team,away_team,attendance,oran
34354,por.1,2022,Casa Pia,Benfica,22253,21.5
32218,por.1,2015,Arouca,Benfica,23540,15.9
32202,por.1,2015,Tondela,Sporting CP,22003,13.7
12932,ger.1,2020,Mainz,VfB Stuttgart,3403,13.6
34585,por.1,2022,Casa Pia,Sporting CP,10509,10.2
8214,eng.2,2021,Huddersfield Town,Reading,146013,8.4


Bu maçların çoğu gerçek. Küçük Portekiz kulüpleri Benfica ve Porto maçlarını daha büyük stadyumlarda oynuyor (Casa Pia'nın Benfica maçında olduğu gibi). Brezilya kulüpleri de büyük maçları daha büyük stadyumlara taşıyor. Pandemiden sonra seyircinin kademeli döndüğü maçlar da bu listede. Bu yüzden göreli bir eşik uygulanmıyor.

**Karar:** Yalnızca fiziksel olarak imkânsız değerler eleniyor. 100 binin üzerindeki seyirci sayıları (1 maç) NaN yapılıyor.

In [16]:
matches.loc[matches["attendance"] > 100_000, "attendance"] = pd.NA

In [17]:
matches.assign(toplam=matches["home_goals"] + matches["away_goals"]).nlargest(3, "toplam")[
    ["league", "season", "home_team", "away_team", "home_goals", "away_goals"]]

,league,season,home_team,away_team,home_goals,away_goals
30054,ned.1,2020,VVV-Venlo,Ajax Amsterdam,0,13
15255,esp.1,2015,Real Madrid,Rayo Vallecano,10,2
6452,eng.2,2017,Bristol City,Hull City,5,5


En yüksek skorlu maçlar (VVV-Venlo 0–13 Ajax, 2020/21; Real Madrid 10–2 Rayo Vallecano, 2015/16) gerçek maçlar. Bunlara dokunulmuyor.

### 1.8 Sonuç sütunu ve kaydetme

In [18]:
matches["result"] = "D"
matches.loc[matches["home_goals"] > matches["away_goals"], "result"] = "H"
matches.loc[matches["home_goals"] < matches["away_goals"], "result"] = "A"

MATCH_COLS = ["league", "season", "match_id", "date", "home_team_id", "home_team",
              "away_team_id", "away_team", "home_goals", "away_goals", "result", "attendance"]
matches = matches[MATCH_COLS].sort_values(["league", "date"]).reset_index(drop=True)
matches.to_csv(PROCESSED / "matches.csv", index=False)
print(matches.shape)
matches["result"].value_counts(normalize=True).round(3)

(36642, 12)


result
H    0.445
A    0.301
D    0.254
Name: proportion, dtype: float64

## 2. Puan tabloları

In [19]:
standings = parse.standings()
print(standings.shape)
standings.head()

(1952, 14)


,league,season,team_id,team,rank,played,wins,draws,losses,goals_for,goals_against,goal_diff,deductions,points
0,eng.1,2015,375,Leicester City,1.0,38.0,23.0,12.0,3.0,68.0,36.0,32.0,0.0,81.0
1,eng.1,2015,359,Arsenal,2.0,38.0,20.0,11.0,7.0,65.0,36.0,29.0,0.0,71.0
2,eng.1,2015,367,Tottenham Hotspur,3.0,38.0,19.0,13.0,6.0,69.0,35.0,34.0,0.0,70.0
3,eng.1,2015,382,Manchester City,4.0,38.0,19.0,9.0,10.0,71.0,41.0,30.0,0.0,66.0
4,eng.1,2015,360,Manchester United,5.0,38.0,19.0,9.0,10.0,49.0,35.0,14.0,0.0,66.0


### 2.1 Tekrar eden kayıtlar, eksik değerler ve veri tipleri

In [20]:
print("tekrar:", standings.duplicated(["league", "season", "team_id"]).sum())
print("eksik:", standings.isna().sum().sum())
STAT_COLS = ["rank", "played", "wins", "draws", "losses", "goals_for",
             "goals_against", "goal_diff", "deductions", "points"]
standings["team_id"] = pd.to_numeric(standings["team_id"])
standings[STAT_COLS] = standings[STAT_COLS].astype(int)
standings.dtypes

tekrar: 0
eksik: 0


league           object
season            int64
team_id           int64
team             object
rank              int64
played            int64
wins              int64
draws             int64
losses            int64
goals_for         int64
goals_against     int64
goal_diff         int64
deductions        int64
points            int64
dtype: object

### 2.2 Tutarlılık kontrolleri

Lig-sezon başına takım sayısı:

In [21]:
standings.pivot_table(index="league", columns="season", values="team_id", aggfunc="size")

season,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
league,,,,,,,,,,,
bra.1,20,20,20,20,20,20,20,20,20,20,20
eng.1,20,20,20,20,20,20,20,20,20,20,20
eng.2,24,24,24,24,24,24,24,24,24,24,24
esp.1,20,20,20,20,20,20,20,20,20,20,20
fra.1,20,20,20,20,20,20,20,20,18,18,18
ger.1,18,18,18,18,18,18,18,18,18,18,18
ita.1,20,20,20,20,20,20,20,20,20,20,20
ned.1,18,18,18,18,18,18,18,18,18,18,18
por.1,18,18,18,18,18,18,18,18,18,18,18


Takım sayıları liglerin gerçek formatlarıyla uyumlu. Örneğin Ligue 1, 2023/24'te 20 takımdan 18 takıma indi.

Her lig-sezonda sıralama 1'den takım sayısına kadar eksiksiz gidiyor mu? Maç tablosundaki takımlarla puan tablosundaki takımlar aynı mı?

In [22]:
rank_ok = standings.groupby(["league", "season"])["rank"].apply(lambda r: sorted(r) == list(range(1, len(r) + 1)))
print("sıralama tutarlı:", rank_ok.all())
print("maçlarda olup puan tablosunda olmayan takım:", len(keys(matches, "home_team_id") - keys(standings)))
print("puan tablosunda olup maçlarda olmayan takım:", len(keys(standings) - keys(matches, "home_team_id")))

sıralama tutarlı: True
maçlarda olup puan tablosunda olmayan takım: 0
puan tablosunda olup maçlarda olmayan takım: 0


Puan, galibiyet ve beraberlikten hesaplanan değerle tutuyor mu?

In [23]:
expected = standings["wins"] * 3 + standings["draws"] - standings["deductions"]
standings.loc[expected != standings["points"], ["league", "season", "team", "wins", "draws", "deductions", "points"]]

,league,season,team,wins,draws,deductions,points
308,eng.2,2018,Birmingham City,14,19,0,52
338,eng.2,2019,Wigan Athletic,15,14,0,47
981,ita.1,2018,Chievo Verona,2,14,0,17
1038,ita.1,2021,Salernitana,7,10,1,31
1348,ned.1,2015,FC Twente,12,7,0,40


Beş satırda tutmuyor. Bunların dördünde gerçek bir puan silme cezası var ama ESPN'in `deductions` alanına girilmemiş: Birmingham City 2018/19 (−9), Wigan Athletic 2019/20 (−12), Chievo 2018/19 (−3) ve FC Twente 2015/16 (−3). Salernitana 2021/22'de ise tersi geçerli: `deductions` alanı 1 gösteriyor ama puan (31) doğru.

**Karar:** `points` alanı resmi puanı doğru veriyor ve analizde o kullanılacak. Güvenilmez olan `deductions` sütunu atılıyor.

In [24]:
standings = standings.drop(columns="deductions").sort_values(["league", "season", "rank"]).reset_index(drop=True)
standings.to_csv(PROCESSED / "standings.csv", index=False)
standings.shape

(1952, 13)

## 3. Gol krallığı

ESPN'in hazır gol krallığı listeleri (`statistics` endpoint'i) her lig-sezon için en çok gol atan 50 oyuncuyu veriyor.

In [25]:
scorers = parse.scorers()
print(scorers.shape)
print("tekrar:", scorers.duplicated(["league", "season", "player_id"]).sum())
print("eksik:", scorers.isna().sum().sum())
for col in ["player_id", "team_id", "goals", "matches"]:
    scorers[col] = pd.to_numeric(scorers[col])

listed = set(zip(scorers["league"], scorers["season"]))
missing = sorted((slug, s) for slug in LEAGUES for s in SEASONS if (slug, s) not in listed)
print(len(listed), "lig-sezonda liste var, eksik olanlar:")
missing

(4550, 9)
tekrar: 0
eksik: 0
91 lig-sezonda liste var, eksik olanlar:


[('bra.1', 2019),
 ('eng.1', 2020),
 ('eng.2', 2018),
 ('esp.1', 2020),
 ('ger.1', 2018),
 ('ita.1', 2018),
 ('ned.1', 2019),
 ('por.1', 2020)]

Sekiz lig-sezonda liste yok. Bunlar ESPN'in o sezonlarda oyuncu istatistiği hiç tutmadığı sezonlar; kadro verisinde de aynı boşluk görülüyor (bkz. 4.2).

Maç verisinde ise her golün olay kaydı var: golü atan oyuncu, takımı ve dakikası. Eksik sezonların gol krallığı bu kayıtlardan hesaplanabilir. Yöntem önce listesi olan 91 lig-sezonda doğrulanıyor.

### 3.1 Maç detaylarından gol sayımı

In [26]:
goal_events = parse.goals()
for col in ["match_id", "team_id", "player_id"]:
    goal_events[col] = pd.to_numeric(goal_events[col])
# Yalnızca temizlenmiş maç tablosundaki maçlar; penaltı atışları gol sayılmaz
goal_events = goal_events[goal_events["match_id"].isin(matches["match_id"]) & ~goal_events["shootout"]]

# Kontrol: her maçta gol olayı sayısı skorla tutuyor mu? (kendi kalesine goller dahil)
per_match = goal_events.groupby("match_id").size().reindex(matches["match_id"], fill_value=0)
score_total = (matches["home_goals"] + matches["away_goals"]).to_numpy()
print("gol olayı sayısı skorla tutan maç oranı:", (per_match.to_numpy() == score_total).mean().round(4))

gol olayı sayısı skorla tutan maç oranı: 0.9979


In [27]:
scored = goal_events[~goal_events["own_goal"]]  # kendi kalesine goller oyuncuya yazılmaz
detail_scorers = (scored.groupby(["league", "season", "player_id"])
                  .agg(player=("player", "first"),
                       goals=("match_id", "size"),
                       # sezon içinde ligde takım değiştirdiyse en çok gol attığı takım
                       team_id=("team_id", lambda t: t.value_counts().idxmax()))
                  .reset_index())


def top_of(df):
    # Her lig-sezonda en çok gol atan oyuncu(lar); eşitlikte hepsi
    return df[df["goals"] == df.groupby(["league", "season"])["goals"].transform("max")]


espn_top, detail_top = top_of(scorers), top_of(detail_scorers)
espn_ids = espn_top.groupby(["league", "season"])["player_id"].apply(set)
detail_ids = detail_top.groupby(["league", "season"])["player_id"].apply(set)
compare = pd.DataFrame({
    "espn_gol": espn_top.groupby(["league", "season"])["goals"].first(),
    "detay_gol": detail_top.groupby(["league", "season"])["goals"].first(),
    "aynı_oyuncu": pd.Series({k: ids == detail_ids.get(k) for k, ids in espn_ids.items()}),
}).dropna().astype({"aynı_oyuncu": bool})
print("gol sayısı aynı:", (compare["espn_gol"] == compare["detay_gol"]).sum(), "/", len(compare))
print("gol kralı aynı oyuncu(lar):", compare["aynı_oyuncu"].sum(), "/", len(compare))
compare[(compare["espn_gol"] != compare["detay_gol"]) | ~compare["aynı_oyuncu"]]

gol sayısı aynı: 83 / 91
gol kralı aynı oyuncu(lar): 87 / 91


espn_gol  detay_gol  aynı_oyuncu
eng.2 2019      26.0         27        False
      2020      32.0         31         True
      2022      28.0         27         True
fra.1 2019      19.0         18        False
ned.1 2018      27.0         28        False
      2022      20.0         19        False
      2024      27.0         24         True
por.1 2021      26.0         23         True

Maçların %99,8'inde gol olayı sayısı skorla tutuyor. Detaylardan bulunan gol kralı, 91 lig-sezonun 87'sinde (%96) ESPN listesindekiyle aynı oyuncu; 83'ünde gol sayısı da birebir aynı. Farkların çoğu 1 gol. Yarışın çok yakın olduğu 4 sezonda bu 1 gol, gol kralının kim olduğunu değiştiriyor. En büyük farklar Eredivisie 2024/25'te (27'ye 24) ve Primeira Liga 2021/22'de (26'ya 23). Primeira Liga'da ESPN listesi Darwin Núñez'in gerçek gol sayısını (26) veriyor. Yani detaylardan yapılan sayım birkaç golü kaçırabiliyor, ama gol kralını büyük çoğunlukla doğru buluyor.

**Karar:** 91 lig-sezonda ESPN listesi, eksik 8 lig-sezonda maç detaylarından hesaplanan gol krallığı kullanılıyor. `source` sütunu hangi kaynağın kullanıldığını gösteriyor. Eşitlik durumunda bütün gol kralları tutuluyor.

In [28]:
filled = detail_top[[k in missing for k in zip(detail_top["league"], detail_top["season"])]]
top_scorers = pd.concat([espn_top.assign(source="espn_list"), filled.assign(source="match_details")])

# Takım adı puan tablosundan; bulunamazsa gol kralının takımı o sezon ligde değil demektir
team_names = standings.set_index(["league", "season", "team_id"])["team"]
top_scorers["team"] = top_scorers.set_index(["league", "season", "team_id"]).index.map(team_names)
print("takımı puan tablosunda bulunamayan gol kralı:", top_scorers["team"].isna().sum())
print("gol kralı olan lig-sezon:", top_scorers.groupby(["league", "season"]).ngroups)
top_scorers.groupby(["league", "season"]).size().value_counts().rename("gol kralı sayısı → lig-sezon")

takımı puan tablosunda bulunamayan gol kralı: 0
gol kralı olan lig-sezon: 99


1    85
2    11
3     3
Name: gol kralı sayısı → lig-sezon, dtype: int64

In [29]:
TOP_COLS = ["league", "season", "player_id", "player", "team_id", "team", "goals", "source"]
top_scorers = top_scorers[TOP_COLS].sort_values(["league", "season"]).reset_index(drop=True)
top_scorers.to_csv(PROCESSED / "top_scorers.csv", index=False)
top_scorers.shape

(116, 8)

## 4. Oyuncular (kadrolar)

In [30]:
players_raw = parse.players()
print(players_raw.shape)
players_raw.head()

(63006, 10)


,league,season,team_id,team,player_id,player,date_of_birth,citizenship,position,appearances
0,eng.1,2015,318,Swansea City,77403,Lukasz Fabianski,1985-04-18T08:00Z,Poland,Goalkeeper,37.0
1,eng.1,2015,318,Swansea City,133993,Kristoffer Nordfeldt,1989-06-23T07:00Z,Sweden,Goalkeeper,1.0
2,eng.1,2015,318,Swansea City,213852,Josh Vickers,1995-12-01T08:00Z,England,Goalkeeper,0.0
3,eng.1,2015,318,Swansea City,102588,Angel Rangel,1982-11-28T08:00Z,Spain,Defender,23.0
4,eng.1,2015,318,Swansea City,107571,Neil Taylor,1989-02-07T08:00Z,Wales,Defender,34.0


### 4.1 Tekrar eden kayıtlar ve eksik değerler

In [31]:
print("tekrar:", players_raw.duplicated(["league", "season", "team_id", "player_id"]).sum())
players_raw.isna().sum()

tekrar: 0


league              0
season              0
team_id             0
team                0
player_id           0
player              0
date_of_birth     102
citizenship       890
position            1
appearances      9694
dtype: int64

`appearances` (o sezonki lig maçı sayısı) yaklaşık 9.700 oyuncuda eksik. Bu alan, ESPN'de istatistiği olmayan oyuncularda boş geliyor. Bu oyuncuların büyük kısmı hiç oynamamış gençler. Doğum tarihi, uyruk ve pozisyondaki eksikler ilerideki adımlarda ele alınıyor.

**Karar:** İstatistiği olmayan oyuncu o sezon hiç oynamamış kabul ediliyor (`appearances = 0`). Bu kararın takım düzeyinde doğru olup olmadığı bir sonraki adımda kontrol ediliyor.

### 4.2 Maç sayısı verisinin kapsamı

Her maçta sahada en az 11 oyuncu olduğu için bir takımın oyuncularının toplam maç sayısı en az 11 × oynanan maç olmalı. Bu koşulu sağlamayan takım-sezonlarda maç sayısı verisi eksik demektir. Tablo, her lig-sezonda koşulu sağlayan takımların oranını gösteriyor:

In [32]:
players = players_raw.copy()
players["appearances"] = players["appearances"].fillna(0).astype(int)

coverage = (players.groupby(["league", "season", "team_id"])["appearances"].sum().to_frame("player_apps")
            .join(standings.set_index(["league", "season", "team_id"])["played"]))
coverage["complete"] = coverage["player_apps"] >= 11 * coverage["played"]
coverage["complete"].groupby(["league", "season"]).mean().unstack().round(2)

season,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
league,,,,,,,,,,,
bra.1,1.00,0.95,1.0,1.0,0.00,0.75,1.0,1.00,1.0,1.0,1.0
eng.1,1.00,1.00,1.0,1.0,1.00,0.00,1.0,1.00,1.0,1.0,1.0
eng.2,1.00,1.00,1.0,0.0,1.00,1.00,1.0,0.75,1.0,1.0,1.0
esp.1,1.00,1.00,1.0,1.0,1.00,0.00,1.0,1.00,1.0,1.0,1.0
fra.1,1.00,1.00,1.0,1.0,1.00,0.90,0.0,1.00,1.0,1.0,1.0
ger.1,1.00,1.00,1.0,0.0,1.00,1.00,1.0,1.00,1.0,1.0,1.0
ita.1,1.00,1.00,1.0,0.0,1.00,1.00,1.0,0.85,0.0,1.0,1.0
ned.1,0.94,1.00,1.0,1.0,0.00,0.72,1.0,1.00,1.0,1.0,1.0
por.1,1.00,1.00,1.0,1.0,0.94,0.00,1.0,1.00,1.0,1.0,1.0


On lig-sezonda veri neredeyse tamamen eksik:

- **ESPN oyuncu istatistiği hiç tutmamış (8 lig-sezon):** Premier League, La Liga ve Primeira Liga 2020/21; Championship, Bundesliga ve Serie A 2018/19; Eredivisie 2019/20; Brasileirão 2019. Bunlar gol krallığı listesi eksik olan sezonlarla aynı.
- **Kadro listesi boş (2 lig-sezon):** Ligue 1 2021/22 ve Serie A 2023/24. Bu sezonlarda takım başına yalnızca 1 oyuncu listelenmiş.

Bunların dışında birkaç takım-sezonda da kısmi eksik var.

**Karar:** Kadro tanımı (en az 1 lig maçına çıkan oyuncular) bu takım-sezonlarda uygulanamıyor. Bu takım-sezonlar Soru 2'nin analizinden çıkarılıyor ve `players.csv`'de `team_complete = False` olarak işaretleniyor.

Not: Championship ve Eredivisie'de bazı oyuncuların maç sayısı takımın maç sayısını 1–2 aşıyor; play-off maçları da sayılmış görünüyor. Bunun kadro tanımına etkisi ihmal edilebilir.

In [33]:
players = players.join(coverage["complete"].rename("team_complete"), on=["league", "season", "team_id"])
print("eksik takım-sezon:", (~coverage["complete"]).sum(), "/", len(coverage))

eksik takım-sezon: 222 / 1952


### 4.3 Veri tipleri ve yaş

Yaş, sezon ortasına göre hesaplanıyor: Avrupa liglerinde sezonun ikinci yılının 1 Ocak'ı (2019/20 için 1 Ocak 2020), Brasileirão'da sezon yılının 1 Temmuz'u. Kadrodaki `age` alanı kullanılmıyor, çünkü oyuncunun bugünkü yaşını veriyor.

In [34]:
players["date_of_birth"] = pd.to_datetime(players["date_of_birth"], utc=True)
calendar = players["league"].map(lambda slug: LEAGUES[slug].get("calendar_year", False))
mid_year = players["season"] + (~calendar).astype(int)
mid_season = pd.to_datetime(mid_year.astype(str) + calendar.map({True: "-07-01", False: "-01-01"}), utc=True)
players["age"] = (mid_season - players["date_of_birth"]).dt.days / 365.25
players.loc[players["appearances"] >= 1, "age"].describe().round(1)

count    46001.0
mean        25.8
std          4.6
min          8.2
25%         22.3
50%         25.5
75%         29.0
max         77.4
Name: age, dtype: float64

### 4.4 Aykırı değerler: hatalı doğum tarihleri

En küçük yaş 8, en büyük yaş 77 çıkıyor. Bu iki uç değerin ikisi de oynamış bir futbolcu için imkânsız.

In [35]:
odd_age = players[(players["appearances"] >= 1) & ((players["age"] < 16) | (players["age"] > 50))]
print(len(odd_age), "oyuncu-sezon")
odd_age.sort_values("age")[["league", "season", "team", "player", "date_of_birth", "appearances", "age"]].round(1).head(12)

91 oyuncu-sezon


,league,season,team,player,date_of_birth,appearances,age
7303,eng.2,2015,Leeds United,Luke Murphy,2007-10-15 07:00:00+00:00,36,8.2
15769,ger.1,2015,Hannover 96,Oliver Sorg,2007-07-31 07:00:00+00:00,20,8.4
7784,eng.2,2016,Burton Albion,Luke Murphy,2007-10-15 07:00:00+00:00,19,9.2
36140,fra.1,2016,Bordeaux,Vukasin Jovanovic,2007-09-30 07:00:00+00:00,8,9.3
7332,eng.2,2015,Bolton Wanderers,Liam Trotter,2006-04-28 07:00:00+00:00,22,9.7
21629,esp.1,2015,Atlético Madrid,Augusto Fernández,2006-01-06 08:00:00+00:00,28,10.0
21964,esp.1,2015,Espanyol,Mamadou Sylla,2005-12-05 08:00:00+00:00,14,10.1
25,eng.1,2015,Swansea City,Alberto Paloschi,2005-11-21 08:00:00+00:00,10,10.1
8514,eng.2,2017,Burton Albion,Luke Murphy,2007-10-15 07:00:00+00:00,38,10.2
42593,ned.1,2017,Ajax Amsterdam,Luis Orejuela,2007-08-11 07:00:00+00:00,1,10.4


Listede açıkça hatalı doğum tarihleri var. Yaya Touré (gerçekte 1983 doğumlu) 2005, Fernando Torres (1984) 2004, Simon Francis (1985) 2005 doğumlu görünüyor ve bu oyuncular onlarca maça çıkmış. Marítimo'nun Brito'su ise 77 yaşında görünüyor. Öte yandan listede gerçekten 15 yaşında olan birkaç oyuncu da var; örneğin 2022/23'te Lamine Yamal ve Ethan Nwaneri. Bu oyuncuların hepsi 1–7 maça çıkmış.

**Karar:** Sezon ortasında 16 yaşından küçük ya da 50 yaşından büyük görünen oyuncuların doğum tarihi hatalı kabul ediliyor ve yaşları NaN yapılıyor. Bu kural birkaç gerçek 15 yaşındaki oyuncuyu da dışarıda bırakıyor. Ama bu oyuncular çok az maça çıktığı için takım ortalamalarına etkileri ihmal edilebilir. Makul bir yaşa denk gelen hatalı doğum tarihleri bu yöntemle yakalanamıyor; bu README'de sınırlılık olarak belirtilecek.

In [36]:
players.loc[(players["age"] < 16) | (players["age"] > 50), "age"] = np.nan
players.loc[players["appearances"] >= 1, "age"].describe().round(1)

count    45910.0
mean        25.9
std          4.6
min         16.0
25%         22.3
50%         25.5
75%         29.0
max         44.7
Name: age, dtype: float64

### 4.5 Tutarsız kategoriler: pozisyon ve uyruk

In [37]:
players["position"].value_counts(dropna=False)

position
Defender      20929
Midfielder    19678
Forward       14912
Goalkeeper     7486
None              1
Name: count, dtype: int64

In [38]:
# Her ligde en sık görülen uyruk, LEAGUES içindeki ülke adıyla aynı yazılmış mı?
pd.DataFrame({
    "en sık uyruk": players.groupby("league")["citizenship"].agg(lambda s: s.value_counts().index[0]),
    "lig ülkesi": pd.Series({slug: info["country"] for slug, info in LEAGUES.items()}),
})

,en sık uyruk,lig ülkesi
bra.1,Brazil,Brazil
eng.1,England,England
eng.2,England,England
esp.1,Spain,Spain
fra.1,France,France
ger.1,Germany,Germany
ita.1,Italy,Italy
ned.1,Netherlands,Netherlands
por.1,Portugal,Portugal


Pozisyonlar dört kategoride ve tutarlı (1 oyuncunun pozisyonu eksik). Uyruk adları lig ülkeleriyle aynı yazılıyor, bu yüzden yerli/yabancı ayrımı doğrudan yapılabiliyor.

### 4.6 Yabancı oyuncu

Onaylanan tanım:

- İngiliz liglerinde England, Wales, Scotland ve Northern Ireland yerli sayılıyor; Republic of Ireland yabancı.
- Diğer liglerde uyruk lig ülkesiyle aynıysa yerli, farklıysa yabancı.
- Uyruğu eksik oyuncular yabancı oranının paydasına girmiyor.

Bu oran kural statüsünü değil, uyruk çeşitliliğini ölçüyor.

In [39]:
UK = {"England", "Wales", "Scotland", "Northern Ireland"}


def domestic_countries(slug):
    return UK if slug.startswith("eng.") else {LEAGUES[slug]["country"]}


players["is_foreign"] = pd.array(
    [pd.NA if pd.isna(c) else c not in domestic_countries(slug)
     for slug, c in zip(players["league"], players["citizenship"])],
    dtype="boolean")
players.loc[players["appearances"] >= 1].groupby("league")["is_foreign"].mean().round(2).sort_values()

league
bra.1    0.14
eng.2    0.38
esp.1    0.42
ned.1     0.5
ger.1    0.57
fra.1    0.57
ita.1    0.61
por.1    0.61
eng.1    0.62
Name: is_foreign, dtype: Float64

### 4.7 Kaydetme

Tablo, hiç oynamayanlar dahil bütün kadro listesini içeriyor. Soru 2'deki kadro tanımı analizde `appearances >= 1` ve `team_complete` filtreleriyle uygulanacak.

In [40]:
PLAYER_COLS = ["league", "season", "team_id", "team", "player_id", "player", "date_of_birth", "age",
               "citizenship", "is_foreign", "position", "appearances", "team_complete"]
players["date_of_birth"] = players["date_of_birth"].dt.date
players["age"] = players["age"].round(2)
players = players[PLAYER_COLS].sort_values(["league", "season", "team_id", "player"]).reset_index(drop=True)
players.to_csv(PROCESSED / "players.csv", index=False)
players.shape

(63006, 13)

## Özet

| Tablo | Uygulanan temizlik |
|---|---|
| `matches.csv` | Play-off ve baraj maçları ile tamamlanmamış 206 maç çıkarıldı. Pandemi dışındaki 0 seyirci değerleri ve 100 bini aşan 1 değer NaN yapıldı. `result` sütunu eklendi (H/D/A). |
| `standings.csv` | Güvenilmez `deductions` sütunu atıldı. Resmi puan `points` sütununda. |
| `top_scorers.csv` | 91 lig-sezonda ESPN listesi, 8 lig-sezonda maç detaylarından hesaplanan gol krallığı kullanıldı. Eşitlikte bütün gol kralları tutuldu. |
| `players.csv` | Maç sayısı eksikse 0 kabul edildi. Maç sayısı verisi eksik takım-sezonlar `team_complete = False` olarak işaretlendi. 16–50 aralığı dışındaki yaşlar NaN yapıldı. `is_foreign` sütunu eklendi. |